# Process one complete UCF101 video: Biking (safe pilot)

This notebook processes **one complete video only**, sampling every 4th frame. It uses CPU inference intentionally because GPU/CUDA-related kernel restarts were observed during the earlier extraction attempt. It does not train or modify ViTPose.

## What it saves
- `data/processed/keypoints_single_tracked/`: one `.npz` file containing the longest-lived tracked person's sampled pose sequence.
- `outputs/pose_videos/`: an annotated MP4 showing skeletons on sampled frames.
- `data/processed/tracked_extraction_report_one_video.csv`: a small summary report.

Run cells from top to bottom, **one at a time**. Do not start the 3-video or full-dataset extraction notebook while this pilot is running. The full video is read sequentially; separate JPEG files for every frame are not created.

## Cell 1 — Import code and load the existing checkpoint

This loads the checkpoint on CPU first and transfers only the model weights into the model. CPU is for diagnosis/stability; once the full-video test works, we can benchmark GPU inference separately.

In [1]:
from pathlib import Path
import sys, gc, re, hashlib, time
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose, get_affine_transform, decode_heatmaps, draw_skeleton_on,
    IMG_H, IMG_W, IMG_MEAN, IMG_STD
)

# Deliberately use CPU for this first complete-video pilot.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"

assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
saved_epoch = checkpoint.get("epoch", "unknown")
del checkpoint
gc.collect()
print("ViTPose loaded; saved epoch:", saved_epoch, flush=True)

print("Loading YOLO detector...", flush=True)
if YOLO_PATH.is_file():
    detector = YOLO(str(YOLO_PATH))
else:
    # Ultralytics downloads the pretrained detector if it is not locally available.
    detector = YOLO("yolo11n.pt")
print("YOLO loaded.", flush=True)
print("Device for this pilot:", DEVICE)

Loading ViTPose checkpoint on CPU...
ViTPose loaded; saved epoch: 209
Loading YOLO detector...
YOLO loaded.
Device for this pilot: cpu


## Cell 2 — Select exactly one Biking video

This uses the manifest you already generated and selects the first video whose label is `Biking`. Confirm the printed path is the expected file before running the extraction cell.

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
required_cols = {"video_path", "label"}
assert required_cols.issubset(manifest.columns), f"Manifest needs columns: {required_cols}"

biking_videos = manifest[
    manifest["label"].astype(str).str.casefold().eq("biking")
].reset_index(drop=True)

print("Biking videos in manifest:", len(biking_videos))
assert len(biking_videos) > 0, "No Biking label found in manifest."

VIDEO_PATH = Path(str(biking_videos.iloc[0]["video_path"]))
VIDEO_LABEL = str(biking_videos.iloc[0]["label"])
assert VIDEO_PATH.is_file(), f"Video not found: {VIDEO_PATH}"

FRAME_STEP = 4              # process original frame 0, 4, 8, 12, ...
DETECTION_CONF = 0.25
MAX_PEOPLE_PER_FRAME = 2    # limit CPU workload during pilot
POSE_THRESH = 0.10

KEYPOINT_DIR = PROJECT_DIR / "data" / "processed" / "keypoints_single_tracked"
VIDEO_OUT_DIR = PROJECT_DIR / "outputs" / "pose_videos"
KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_OUT_DIR.mkdir(parents=True, exist_ok=True)

video_hash = hashlib.sha1(str(VIDEO_PATH.resolve()).encode("utf-8")).hexdigest()[:10]
safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", VIDEO_LABEL).strip("_")
NPZ_PATH = KEYPOINT_DIR / f"{safe_label}__{VIDEO_PATH.stem}__{video_hash}.npz"
ANNOTATED_PATH = VIDEO_OUT_DIR / f"{VIDEO_PATH.stem}_tracked_pose.mp4"
REPORT_PATH = PROJECT_DIR / "data" / "processed" / "tracked_extraction_report_one_video.csv"

print("Selected label:", VIDEO_LABEL)
print("Selected video:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.is_file())
print("Frame sampling step:", FRAME_STEP)
print("Keypoint output:", NPZ_PATH)
print("Annotated video output:", ANNOTATED_PATH)

Biking videos in manifest: 134
Selected label: Biking
Selected video: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports\Biking\v_Biking_g01_c01.avi
Video exists: True
Frame sampling step: 4
Keypoint output: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_single_tracked\Biking__v_Biking_g01_c01__b5e8eaca1f.npz
Annotated video output: C:\Users\admin\ViTPose_Action_Recognition\outputs\pose_videos\v_Biking_g01_c01_tracked_pose.mp4


## Cell 3 — Define pose preprocessing

This function uses the same crop transform and normalization expected by the original model. It runs one detected person's crop through ViTPose and decodes 17 joints.

In [3]:
def pose_for_box(frame_rgb, bbox):
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = np.clip(x1, 0, w - 1)
    x2 = np.clip(x2, 0, w)
    y1 = np.clip(y1, 0, h - 1)
    y2 = np.clip(y2, 0, h)
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, trans, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()
    keypoints = decode_heatmaps(heatmaps, center, scale)
    return keypoints.astype(np.float32)

print("Pose preprocessing function is ready.")

Pose preprocessing function is ready.


## Cell 4 — Process the complete video

This is the main cell. It reads every frame sequentially, but runs YOLO tracking and ViTPose only on every 4th frame. It saves sampled frames to the annotated MP4 and stores the longest-lived track in one `.npz` file.

A short video should take much less time than extracting the entire UCF101 dataset. CPU inference may be slower than GPU inference, but it avoids relying on CUDA while we debug the full-video pipeline.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), f"OpenCV cannot open video: {VIDEO_PATH}"
fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
if fps <= 0:
    fps = 25.0
frame_count_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

# The output contains only sampled frames, so divide FPS by FRAME_STEP to roughly
# preserve the original playback duration.
writer = cv2.VideoWriter(
    str(ANNOTATED_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    max(fps / FRAME_STEP, 1.0),
    (width, height)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not create output video: {ANNOTATED_PATH}")

observations = []
frame_i = 0
sampled_i = 0
start_time = time.time()

# Start a fresh tracker state for this video.
predictor = getattr(detector, "predictor", None)
if predictor is not None and hasattr(predictor, "trackers"):
    for tracker_obj in predictor.trackers:
        if hasattr(tracker_obj, "reset"):
            tracker_obj.reset()

try:
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if frame_i % FRAME_STEP == 0:
            # Tracking on CPU for the pilot; only person class (COCO class 0).
            result = detector.track(
                frame_bgr,
                persist=True,
                tracker="bytetrack.yaml",
                classes=[0],
                conf=DETECTION_CONF,
                device="cpu",
                verbose=False
            )[0]

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            visual_rgb = frame_rgb.copy()

            if result.boxes is not None and len(result.boxes) > 0:
                xyxy = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                ids = (
                    result.boxes.id.cpu().numpy().astype(np.int64)
                    if result.boxes.id is not None
                    else np.full(len(xyxy), -1, dtype=np.int64)
                )

                areas = np.maximum(0, xyxy[:, 2] - xyxy[:, 0]) * np.maximum(0, xyxy[:, 3] - xyxy[:, 1])
                keep = np.argsort(areas)[::-1][:MAX_PEOPLE_PER_FRAME]

                for det_idx in keep:
                    bbox = xyxy[det_idx].astype(np.float32)
                    kps = pose_for_box(frame_rgb, bbox)
                    if kps is None:
                        continue

                    track_id = int(ids[det_idx])
                    observations.append({
                        "frame_idx": int(frame_i),
                        "track_id": track_id,
                        "keypoints": kps,
                        "box": bbox,
                        "det_score": float(confs[det_idx]),
                    })

                    visual_rgb = draw_skeleton_on(visual_rgb, kps, thresh=POSE_THRESH)
                    x1, y1, x2, y2 = map(int, bbox)
                    cv2.rectangle(visual_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label_text = f"Person ID {track_id}" if track_id >= 0 else "Person ID unknown"
                    cv2.putText(
                        visual_rgb, label_text,
                        (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
                    )

            writer.write(cv2.cvtColor(visual_rgb, cv2.COLOR_RGB2BGR))
            sampled_i += 1

            if sampled_i % 5 == 0:
                print(
                    f"Sampled frames processed: {sampled_i}; "
                    f"original frame index: {frame_i}; "
                    f"pose observations: {len(observations)}",
                    flush=True
                )

        frame_i += 1
finally:
    cap.release()
    writer.release()

print("\nVideo processing loop finished.")
print("Original frames read:", frame_i)
print("Video frame count metadata:", frame_count_meta)
print("Sampled frames processed:", sampled_i)
print("Person pose observations:", len(observations))
print("Elapsed seconds:", round(time.time() - start_time, 1))
print("Annotated video exists:", ANNOTATED_PATH.exists())

if not observations:
    raise RuntimeError("No pose observations were saved; inspect detection output or choose another video.")

## Cell 5 — Select the longest-lived track and save pose data

Tracking may return multiple people. For this pilot, choose the track observed in the most distinct sampled frames, breaking ties by total bounding-box area. This is a baseline rule, so inspect the annotated video to ensure it selected the cyclist/action performer.

In [ ]:
assigned = [o for o in observations if o["track_id"] >= 0]

if assigned:
    candidate_ids = sorted({o["track_id"] for o in assigned})
    track_stats = {}
    for tid in candidate_ids:
        rows = [o for o in assigned if o["track_id"] == tid]
        distinct_frames = len({o["frame_idx"] for o in rows})
        total_area = float(sum(
            max(0, o["box"][2] - o["box"][0]) * max(0, o["box"][3] - o["box"][1])
            for o in rows
        ))
        track_stats[tid] = (distinct_frames, total_area)
    target_track_id = max(candidate_ids, key=lambda tid: track_stats[tid])
    selected = [o for o in assigned if o["track_id"] == target_track_id]
    tracker_fallback = False
else:
    # If IDs are unavailable, keep the largest detection per sampled frame and flag it.
    by_frame = {}
    for o in observations:
        by_frame.setdefault(o["frame_idx"], []).append(o)
    selected = [
        max(rows, key=lambda o: (o["box"][2] - o["box"][0]) * (o["box"][3] - o["box"][1]))
        for rows in by_frame.values()
    ]
    target_track_id = -1
    tracker_fallback = True

selected.sort(key=lambda o: o["frame_idx"])

pose_data = {
    "keypoints": np.stack([o["keypoints"] for o in selected]).astype(np.float32),
    "boxes": np.stack([o["box"] for o in selected]).astype(np.float32),
    "frame_indices": np.asarray([o["frame_idx"] for o in selected], dtype=np.int32),
    "track_ids": np.asarray([o["track_id"] for o in selected], dtype=np.int64),
    "detection_scores": np.asarray([o["det_score"] for o in selected], dtype=np.float32),
    "target_track_id": np.asarray(target_track_id, dtype=np.int64),
    "label": np.asarray(VIDEO_LABEL),
    "video_path": np.asarray(str(VIDEO_PATH)),
    "fps": np.asarray(fps, dtype=np.float32),
    "frame_step": np.asarray(FRAME_STEP, dtype=np.int32),
    "total_video_frames": np.asarray(frame_i, dtype=np.int64),
}

# Write through a temporary file, then rename, so a partial write doesn't look complete.
temp_path = NPZ_PATH.with_name(NPZ_PATH.stem + ".tmp.npz")
np.savez_compressed(temp_path, **pose_data)
temp_path.replace(NPZ_PATH)

report = pd.DataFrame([{
    "video_path": str(VIDEO_PATH),
    "label": VIDEO_LABEL,
    "status": "ok",
    "total_video_frames": int(frame_i),
    "sampled_frames": int(sampled_i),
    "pose_observations_all_tracks": int(len(observations)),
    "selected_track_observations": int(len(selected)),
    "target_track_id": int(target_track_id),
    "tracker_fallback": bool(tracker_fallback),
    "frame_step": int(FRAME_STEP),
    "fps": float(fps),
    "seconds": round(time.time() - start_time, 1),
    "keypoint_path": str(NPZ_PATH),
    "annotated_video_path": str(ANNOTATED_PATH),
}])
report.to_csv(REPORT_PATH, index=False)

print("Saved keypoints:", NPZ_PATH.resolve())
print("Saved annotated video:", ANNOTATED_PATH.resolve())
print("Saved report:", REPORT_PATH.resolve())
print("Selected track ID:", target_track_id)
print("Selected pose observations:", len(selected))
print("Keypoints array shape:", pose_data["keypoints"].shape, "(T, 17, 3)")
print("Frame indices:", pose_data["frame_indices"][:10], "...")

## Cell 6 — Verify the saved files

Run this cell and confirm that the `.npz`, annotated MP4 and report exist. Open the MP4 using Jupyter's file browser or Windows media player. Review whether the skeleton follows the cyclist and whether the selected person ID remains consistent.

If this pilot succeeds, do not immediately process all 13,320 videos. Next, select a balanced five-class subset and add video-level train/validation/test splits before extracting the larger subset.

In [ ]:
print("Keypoint file exists:", NPZ_PATH.is_file(), "| MB:", round(NPZ_PATH.stat().st_size / (1024**2), 3) if NPZ_PATH.exists() else "n/a")
print("Annotated video exists:", ANNOTATED_PATH.is_file(), "| MB:", round(ANNOTATED_PATH.stat().st_size / (1024**2), 3) if ANNOTATED_PATH.exists() else "n/a")
print("Report exists:", REPORT_PATH.is_file())

with np.load(NPZ_PATH, allow_pickle=False) as saved:
    for key in saved.files:
        value = saved[key]
        print(key, getattr(value, "shape", None), getattr(value, "dtype", None))

print("\nReport:")
display(pd.read_csv(REPORT_PATH))